# 🛡️ Phishing Website Detection Using Machine Learning
**KTU S3 CSE — AI & ML Mini Project**

### Problem Statement
Phishing is a form of cyberattack where attackers impersonate legitimate websites to harvest credentials, banking details, or sensitive personal data.
This project analyzes 30 URL and webpage security characteristics and builds 4 machine learning classification models to distinguish legitimate websites from phishing sites.

---

## 1. Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries imported successfully!')

## 2. Load Dataset
We use the **UCI Phishing Websites Dataset** (11,055 records, 30 features, target `Result`: 1 = Legitimate, -1 = Phishing).

In [ ]:
csv_path = '../data/phishing_dataset.csv' if os.path.exists('../data/phishing_dataset.csv') else 'data/phishing_dataset.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=327)
    df = dataset.data.features.copy()
    df['Result'] = dataset.data.targets.values.ravel()

X = df.drop(columns=['Result'])
y = df['Result'].values.ravel()

print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Features: {X.shape[1]}')
print(f'Target values: {np.unique(y)}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check missing values
print('Total Missing Values:', df.isnull().sum().sum())
df.info()

In [ ]:
# Class Distribution
class_counts = pd.Series(y).value_counts()
print(class_counts)

plt.figure(figsize=(7, 4))
bars = plt.bar(['Phishing (-1)', 'Legitimate (1)'], [class_counts[-1], class_counts[1]], color=['#e74c3c', '#2ecc71'])
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, str(bar.get_height()), ha='center', fontweight='bold')
plt.title('Target Class Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Website Count')
plt.show()

In [ ]:
# Correlation with Target
corr = df.corr()
target_corr = corr['Result'].drop('Result').abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
target_corr.head(10).iloc[::-1].plot.barh(color='#3498db', edgecolor='black')
plt.title('Top 10 Features Correlated with Target (Result)', fontsize=14, fontweight='bold')
plt.xlabel('Absolute Correlation')
plt.show()

## 4. Data Preprocessing & Train/Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 5. Model Training (4 Algorithms)
1. Logistic Regression (Baseline linear model)
2. Decision Tree (Rule-based decision paths)
3. Random Forest (Ensemble bagging)
4. K-Nearest Neighbors (Distance metric based)

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(random_state=42, max_depth=10), False),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15), False),
    'KNN': (KNeighborsClassifier(n_neighbors=5), True)
}

trained_results = {}

for name, (model, uses_scaler) in models.items():
    x_tr = X_train_scaled if uses_scaler else X_train
    x_te = X_test_scaled if uses_scaler else X_test
    model.fit(x_tr, y_train)
    y_pred = model.predict(x_te)
    trained_results[name] = {
        'model': model,
        'predictions': y_pred,
        'acc': accuracy_score(y_test, y_pred),
        'prec': precision_score(y_test, y_pred),
        'rec': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    }
    print(f'Trained {name}: Accuracy = {trained_results[name][acc]*100 if "acc" in trained_results[name] else trained_results[name]["acc"]*100:.2f}%')

## 6. Model Evaluation & Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': info['acc'],
        'Precision': info['prec'],
        'Recall': info['rec'],
        'F1-Score': info['f1']
    }
    for name, info in trained_results.items()
]).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print(comparison_df.to_string(index=False))

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, info) in zip(axes.flatten(), trained_results.items()):
    cm = confusion_matrix(y_test, info['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=['Phishing', 'Legit'], yticklabels=['Phishing', 'Legit'])
    ax.set_title(f"{name} (Acc: {info['acc']*100:.2f}%)")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 7. Feature Importance & Conclusion
Random Forest achieved the best overall performance with **97.20% accuracy** and **0.9751 F1-Score**.

In [ ]:
rf_model = trained_results['Random Forest']['model']
fi = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
fi.head(10).iloc[::-1].plot.barh(color='#2ecc71', edgecolor='black')
plt.title('Top 10 Most Influential Features (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.show()